# KW26 — Colab 보조 추론 워커 · 계정 1 (GPU 만 빌려준다)

이 노트북은 **언어모델 계산만** 한다. 시뮬레이션·DB·기록은 모두 우리 A100 서버에 있고, 이 노트북은 그것들에 접근할 수 없다.

## 실행 방법
1. **런타임 → 런타임 유형 변경 → G4** (없으면 A100/H100, GPU 메모리 40GB 이상).
2. **Secrets 는 쓰지 않는다.** 이 노트북은 계정 1 전용이다(계정 번호가 박혀 있다).
   키는 이 런타임 안에서 만들어지고, 마지막 셀이 찍는 `KW26_PUBKEY` 줄(공개 쪽)을 운영자가 서버에 등록한다.
3. **런타임 → 모두 실행**. 처음에는 설치·모델 내려받기로 15~25분 걸린다.
4. **마지막 셀은 끝나지 않고 계속 돈다. 멈추지 말 것.** 실행 중인 셀이 없으면 Colab 이 약 90분 뒤 세션을 끊는다.
5. Colab 은 연속 **최대 24시간**이다. 끊기면 **모두 실행**을 다시 누른다. 끊겨 있는 동안 시뮬레이션은 멈추지 않는다(느려질 뿐).

## 무엇을 확인하나
- 모델 서버 설정이 우리 A100 과 **한 글자도 다르지 않을 때만** 우리 서버가 일을 준다(노트북도 먼저 스스로 검사한다).
- 마지막 셀이 1분마다 `health {...: True} | tunnel UP | ...` 를 찍으면 정상이다.

In [ ]:
# 1) GPU 확인
import os, subprocess
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version,compute_cap',
                      '--format=csv,noheader,nounits'], capture_output=True, text=True, check=True).stdout.strip()
print(out, '| CPU cores', os.cpu_count())
GPU_NAME, GPU_MIB = out.splitlines()[0].split(',')[0].strip(), int(out.splitlines()[0].split(',')[1])
assert GPU_MIB >= 40000, f'{GPU_NAME} {GPU_MIB}MiB: G4/H100/A100 런타임으로 바꿔 주세요.'
# [2026-10-06] GPU 한 장에 서버 하나. G4 에 서버 둘을 올렸더니(43GB+51GB) 실행 중 쓸 여유가 없어 서버 하나가
# 메모리 부족으로 죽고 자동 재시작도 같은 이유로 실패했다(본런 첫 30분). 둘의 이득은 약 16%뿐이었다(doinggyu 기록).
N_SERVERS = 1
PER_SERVER_MIB = int(GPU_MIB * 0.85)
PORTS = [8000 + i for i in range(N_SERVERS)]
print('model servers:', PORTS, '| per-server budget MiB', PER_SERVER_MIB)

In [ ]:
# 2) 우리 A100 `venv_sgl` 의 정확한 패키지 목록 (sha256 54988d1897e8c47d675cebe8bb3759e7d8925e7953d09200bc59e46747dba43f)
import hashlib, pathlib
FREEZE = 'addict==2.4.0\naiohappyeyeballs==2.7.1\naiohttp==3.14.3\naiosignal==1.4.0\nairportsdata==20260905\nannotated-doc==0.0.5\nannotated-types==0.8.0\nanthropic==1.5.0\nanyio==4.15.1\napache-tvm-ffi==0.1.13.post3\nasttokens==3.0.2\nattrs==26.1.0\nav==16.1.0\nblobfile==3.0.0\nbuild==1.6.1\ncache_dit==1.3.0\ncertifi==2026.7.22\ncffi==2.1.1\ncharset-normalizer==3.5.1\nclick==8.5.0\ncloudpickle==3.1.2\ncompressed-tensors==0.15.0.1\ncryptography==50.0.1\ncuda-bindings==12.9.7\ncuda-pathfinder==1.8.1\ncuda-python==12.9.0\ncuda-tile==1.6.0\ndatasets==5.0.1\ndecorator==5.3.1\ndiffusers==0.37.0\ndill==0.4.1\ndiskcache==5.6.3\ndistro==1.9.0\ndnspython==2.8.0\ndocstring_parser==0.18.0\neinops==0.8.2\nemail-validator==2.3.0\nexecuting==2.2.1\nfastapi==0.141.1\nfilelock==3.32.6\nflash-attn-4==4.0.0b29\nflashinfer-cubin==0.6.7.post2\nflashinfer-python==0.6.7.post2\nfrozenlist==1.8.0\nfsspec==2026.6.0\ngguf==0.19.0\ngoogleapis-common-protos==1.75.3\ngranian==2.8.2\ngrpcio==1.83.1\ngrpcio-health-checking==1.81.1\ngrpcio-reflection==1.81.1\nh11==0.16.0\nhf-xet==1.6.0\nhttpcore==1.0.9\nhttpcore2==2.12.0\nhttpx==0.28.1\nhttpx2==2.12.0\nhuggingface_hub==1.31.0\nhumanize==4.16.0\nid==1.6.1\nidna==3.19\nimageio==2.36.0\nimageio-ffmpeg==0.5.1\nimportlib_metadata==9.0.1\ninteregular==0.3.3\nipython==9.17.1\nipython_pygments_lexers==1.1.1\njedi==0.20.0\nJinja2==3.1.6\njiter==0.16.0\njsonschema==4.26.0\njsonschema-specifications==2025.9.1\nkernels==0.10.0\nkernels-data==0.16.1\nlark==1.3.1\nlazy-loader==0.5\nllguidance==0.7.30\nloguru==0.7.3\nlxml==6.1.3\nmarkdown-it-py==4.2.0\nMarkupSafe==3.0.3\nmatplotlib-inline==0.2.2\nmdurl==0.1.2\nmistral_common==1.11.7\nmodelscope==1.40.0\nmodelscope-hub==0.4.2\nmoviepy==2.2.1\nmpmath==1.3.0\nmsgspec==0.21.1\nmultidict==6.8.0\nmultiprocess==0.70.19\nnest-asyncio==1.6.0\nnetworkx==3.6.1\nninja==1.13.2\nnumpy==2.3.5\nnvidia-cublas-cu12==12.8.4.1\nnvidia-cuda-cupti-cu12==12.8.90\nnvidia-cuda-nvdisasm==13.4.49\nnvidia-cuda-nvrtc-cu12==12.8.93\nnvidia-cuda-runtime-cu12==12.8.90\nnvidia-cudnn-cu12==9.10.2.21\nnvidia-cudnn-frontend==1.28.0\nnvidia-cufft-cu12==11.3.3.83\nnvidia-cufile-cu12==1.13.1.3\nnvidia-curand-cu12==10.3.9.90\nnvidia-cusolver-cu12==11.7.3.90\nnvidia-cusparse-cu12==12.5.8.93\nnvidia-cusparselt-cu12==0.7.1\nnvidia-cutlass-dsl==4.6.2\nnvidia-cutlass-dsl-libs-base==4.6.2\nnvidia-cutlass-dsl-libs-core==4.6.2\nnvidia-cutlass-dsl-libs-cu12==4.6.2\nnvidia-ml-py==13.610.43\nnvidia-nccl-cu12==2.27.5\nnvidia-nvjitlink-cu12==12.8.93\nnvidia-nvshmem-cu12==3.3.20\nnvidia-nvtx-cu12==12.8.90\nopenai==2.6.1\nopenai-harmony==0.0.4\nopencv-python-headless==4.10.0.84\nopentelemetry-api==1.44.0\nopentelemetry-exporter-otlp==1.44.0\nopentelemetry-exporter-otlp-proto-common==1.44.0\nopentelemetry-exporter-otlp-proto-grpc==1.44.0\nopentelemetry-exporter-otlp-proto-http==1.44.0\nopentelemetry-proto==1.44.0\nopentelemetry-sdk==1.44.0\nopentelemetry-semantic-conventions==0.65b0\norjson==3.12.0\noutlines==0.1.11\noutlines_core==0.1.26\npackaging==26.3\npandas==3.0.5\nparso==0.8.7\npartial-json-parser==0.2.1.1.post7\npexpect==4.9.0\npillow==11.3.0\nplatformdirs==4.11.8\nproglog==0.1.12\nprometheus_client==0.26.0\nprompt_toolkit==3.0.53\npropcache==0.5.2\nprotobuf==6.33.6\npsutil==7.2.2\nptyprocess==0.7.0\npure_eval==0.2.4\npy-spy==0.4.2\npyarrow==25.0.1\npyasn1==0.6.4\npybase64==1.5.0\npycountry==26.2.16\npycparser==3.0\npycryptodomex==3.23.0\npydantic==2.13.5\npydantic-extra-types==2.11.1\npydantic_core==2.46.5\nPygments==2.21.0\nPyJWT==2.13.0\npyOpenSSL==26.4.0\npyproject_hooks==1.2.0\npython-dateutil==2.9.0.post0\npython-dotenv==1.2.3\npython-multipart==0.0.32\nPyYAML==6.0.1\npyzmq==27.2.0\nquack-kernels==0.6.4\nreferencing==0.37.0\nregex==2026.9.10\nremote-pdb==2.1.0\nrequests==2.34.2\nrfc3161-client==1.0.8\nrfc8785==0.1.4\nrich==15.0.0\nrpds-py==2026.6.3\nrunai-model-streamer==0.16.1\nsafetensors==0.8.0\nscikit-image==0.25.2\nscipy==1.18.1\nsecuresystemslib==1.5.1\nsentencepiece==0.2.2\nsetproctitle==1.3.7\nsetuptools==84.0.0\nsglang @ git+https://github.com/lkm2835/sglang.git@6757c9f904cdb8ae9028a394a2108d079b9e088c#subdirectory=python\nsglang-kernel==0.4.1\nshellingham==1.5.4\nsigstore==4.5.0\nsigstore-models==0.0.6\nsigstore-rekor-types==0.0.18\nsix==1.17.0\nsmg-grpc-proto==0.4.18\nsmg-grpc-servicer==0.9.1\nsniffio==1.3.1\nsoundfile==0.13.1\nst_attn==0.0.7\nstack-data==0.6.3\nstarlette==1.6.0\nsympy==1.14.0\ntabulate==0.10.0\ntifffile==2026.9.9\ntiktoken==0.14.0\ntimm==1.0.16\ntokenizers==0.22.2\ntomlkit==0.15.1\ntorch==2.9.1\ntorch_c_dlpack_ext==0.1.5\ntorch_memory_saver==0.0.9\ntorchao==0.9.0\ntorchaudio==2.9.1\ntorchcodec==0.9.1\ntorchvision==0.24.1\ntqdm==4.70.0\ntraitlets==5.16.1\ntransformers==5.8.0\ntrimesh==5.1.0\ntriton==3.5.1\ntruststore==0.10.4\ntuf==7.0.1\ntyper==0.27.2\ntyping-inspection==0.4.4\ntyping_extensions==4.16.0\nurllib3==2.7.0\nuvicorn==0.52.4\nuvloop==0.22.1\nvsa==0.0.4\nwatchfiles==1.2.0\nwcwidth==0.8.3\nxatlas==0.0.11\nxgrammar==0.1.32\nxxhash==4.0.1\nyarl==1.24.5\nzipp==4.1.0\n'
assert hashlib.sha256(FREEZE.encode()).hexdigest() == '54988d1897e8c47d675cebe8bb3759e7d8925e7953d09200bc59e46747dba43f'
pathlib.Path('/content/a100_venv_sgl_freeze.txt').write_text(FREEZE)
print('freeze lines', len(FREEZE.splitlines()))

In [ ]:
# 3) 설치 + 모델 내려받기 (이미 되어 있으면 건너뜀)
import os, subprocess, sys, pathlib
VENV = '/content/venv_sgl/bin/python'
MODEL, REVISION, SEED = 'LGAI-EXAONE/EXAONE-4.5-33B-AWQ', '31e6a965d0661bbe4a8b895e22a77f8271772ba0', 762156816
def sh(cmd, check=True, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if check and r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:], sep='\n')
        r.check_returncode()
    return r.stdout
if not os.path.exists(VENV) or subprocess.run([VENV, '-c', 'import sglang, torch, xgrammar']).returncode:
    sh([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
    UV = [sys.executable, '-m', 'uv']
    sh(UV + ['python', 'install', '3.12'])          # A100 venv 는 Python 3.12
    if not os.path.exists(VENV):
        sh(UV + ['venv', '--python', '3.12', '/content/venv_sgl'])
    # 목록이 정확한 전체 목록이므로 의존성을 다시 계산하지 않고(--no-deps) 그대로 설치한다.
    sh(UV + ['pip', 'install', '--no-deps', '--python', VENV, '-r', '/content/a100_venv_sgl_freeze.txt'])
print(sh([VENV, '-c', 'import importlib.metadata as md, torch; '
          'print("sglang", md.version("sglang"), "| torch", torch.__version__, "| transformers", md.version("transformers"), '
          '"| xgrammar", md.version("xgrammar")); print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))']))
HF_HOME = '/content/hf_cache'
os.environ['HF_HOME'] = HF_HOME
sh([VENV, '-c', f"from huggingface_hub import snapshot_download; snapshot_download('{MODEL}', revision='{REVISION}')"],
   env=dict(os.environ, HF_HOME=HF_HOME))
# A100 은 --revision 없이 떴다(서버 정보 revision=null). 같은 커밋을 쓰면서 revision 을 null 로 두려고
# 캐시의 main 을 그 커밋으로 고정하고 서버는 오프라인으로 띄운다.
ref = pathlib.Path(HF_HOME) / 'hub' / ('models--' + MODEL.replace('/', '--')) / 'refs' / 'main'
ref.parent.mkdir(parents=True, exist_ok=True); ref.write_text(REVISION)
assert (ref.parent.parent / 'snapshots' / REVISION).is_dir(), '모델 커밋이 캐시에 없다'
print('install + model ok — main ->', REVISION[:12])

In [ ]:
# 4) A100 과 같은 인자로 모델 서버 시작 + 설정 일치 확인 (이미 떠 있으면 건너뜀)
import json, time, urllib.request
ARGS = ['--model-path', MODEL, '--host', '127.0.0.1', '--tp-size', '1', '--attention-backend', 'triton',
        '--trust-remote-code', '--reasoning-parser', 'qwen3', '--random-seed', str(SEED)]
EXPECTED = {'model_path': 'LGAI-EXAONE/EXAONE-4.5-33B-AWQ', 'served_model_name': 'LGAI-EXAONE/EXAONE-4.5-33B-AWQ', 'revision': None, 'context_length': None, 'grammar_backend': 'xgrammar', 'constrained_json_whitespace_pattern': None, 'constrained_json_disable_any_whitespace': False, 'random_seed': 762156816, 'dtype': 'auto', 'version': '0.0.0.dev11420+g6757c9f90', 'quantization': None, 'kv_cache_dtype': 'auto', 'trust_remote_code': True, 'chat_template': None, 'tokenizer_path': 'LGAI-EXAONE/EXAONE-4.5-33B-AWQ', 'tokenizer_mode': 'auto', 'sampling_defaults': 'model', 'preferred_sampling_params': None, 'json_model_override_args': '{}', 'reasoning_parser': 'qwen3', 'skip_tokenizer_init': False, 'is_embedding': False, 'speculative_algorithm': None, 'enable_deterministic_inference': False}
# 가중치 변환 커널을 즉석 컴파일하므로 venv 의 ninja 와 CUDA nvcc 가 PATH 에 있어야 한다(doinggyu 운영 기록).
SERVER_ENV = dict(os.environ, PATH='/content/venv_sgl/bin:/usr/local/cuda/bin:' + os.environ.get('PATH', ''),
                  CUDA_HOME='/usr/local/cuda', HF_HOME=HF_HOME, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
def healthy(port):
    try:
        return urllib.request.urlopen(f'http://127.0.0.1:{port}/health', timeout=3).status == 200
    except Exception:
        return False
def fraction_for(target_mib):
    # --mem-fraction-static 은 GPU 를 혼자 쓴다고 가정한다. 다른 서버가 쓰는 만큼을 빼고 계산한다.
    free, total = [int(x) for x in sh(['nvidia-smi', '--query-gpu=memory.free,memory.total',
                                       '--format=csv,noheader,nounits']).split(',')]
    assert free - target_mib > 3000, f'GPU 여유 부족: free {free} MiB'
    return round(1 - (free - target_mib) / total, 3)
def check_identity(port):
    info = json.load(urllib.request.urlopen(f'http://127.0.0.1:{port}/get_server_info', timeout=10))
    diff = {k: (v, info.get(k, '<absent>')) for k, v in EXPECTED.items() if info.get(k, '<absent>') != v}
    assert not diff, f'{port} A100 과 설정이 다르다: {diff}'
    return info.get('max_total_num_tokens')
def start_server(port):
    if healthy(port):
        print(port, 'already healthy | KV tokens', check_identity(port)); return
    logpath = f'/content/sglang-{port}.log'
    open(logpath, 'w').close()
    fraction = fraction_for(PER_SERVER_MIB)
    proc = subprocess.Popen([VENV, '-m', 'sglang.launch_server', *ARGS, '--port', str(port),
                             '--mem-fraction-static', str(fraction)],
                            stdout=open(logpath, 'ab'), stderr=subprocess.STDOUT, start_new_session=True, env=SERVER_ENV)
    t0 = time.time()
    while not healthy(port):
        if proc.poll() is not None:
            print(open(logpath, errors='replace').read()[-5000:])
            raise RuntimeError(f'SGLang {port} exited (code {proc.returncode})')
        assert time.time() - t0 < 40 * 60, f'{port} not healthy in 40 min'
        time.sleep(15)
    print(port, f'healthy after {int(time.time() - t0)}s | mem_fraction_static {fraction} | KV tokens', check_identity(port), flush=True)
start_server(PORTS[0])
if len(PORTS) > 1:
    try:
        start_server(PORTS[1])
    except Exception as exc:
        print('두 번째 서버를 못 띄웠다 — 서버 1개로 계속한다:', str(exc)[:300])
        PORTS = PORTS[:1]
print(sh(['nvidia-smi', '--query-gpu=memory.used,memory.total', '--format=csv,noheader']).strip())

In [ ]:
# 5) 우리 서버로 역방향 터널 + 상태 루프.  ★ 이 셀은 끝나지 않는다 — 멈추지 말 것 ★
import pathlib, re
# [2026-10-06] Secrets 를 쓰지 않는다 — 탭이 닫힌 채 이 칸에 오면 Secrets 를 읽지 못해 멈추고(TimeoutException),
# 새 노트북마다 '보안 비밀 액세스' 창이 떠서 사람이 눌러야 했다. 계정 번호는 계정별 노트북에 박혀 있고,
# 키는 이 런타임 안에서 만든다. 비밀 쪽은 Colab 밖으로 나가지 않고 KW26_PUBKEY 줄(공개 쪽)만 운영자가
# 서버에 이 계정 포트로 제한해 등록한다(register_colab_key.sh). 등록 전까지 터널은 5초마다 다시 시도한다.
ACCOUNT = 1
assert 1 <= ACCOUNT <= 6, '계정 번호는 1~6'
ssh_dir = pathlib.Path('/root/.ssh'); ssh_dir.mkdir(mode=0o700, exist_ok=True)
if not (ssh_dir / 'kw26_pool').exists():   # 같은 런타임에서 다시 돌리면 이 키를 그대로 쓴다
    sh(['ssh-keygen', '-q', '-t', 'ed25519', '-N', '', '-C',
        f'kw26-colab-session-{ACCOUNT}-{time.strftime("%Y%m%d%H%M%S")}', '-f', str(ssh_dir / 'kw26_pool')])
print('KW26_PUBKEY', ACCOUNT, (ssh_dir / 'kw26_pool.pub').read_text().strip(), flush=True)
print('key fingerprint:', sh(['ssh-keygen', '-lf', '/root/.ssh/kw26_pool']).strip())
(ssh_dir / 'known_hosts').write_text('[123.37.28.167]:10022 ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIGY7H2+5QX7th9/AMrlHAdkTd8xf/aBH730eKayZTTD4\n')
# 이 키는 우리 서버에서 셸이 막혀 있고, 이 계정 번호의 포트 두 개(180N1, 180N2)만 열 수 있다.
forwards = ' '.join(f'-R 127.0.0.1:180{ACCOUNT}{i + 1}:127.0.0.1:{p}' for i, p in enumerate(PORTS))
subprocess.run(['pkill', '-f', '/content/tunnel.sh']); subprocess.run(['pkill', '-f', 'ssh -N -i /root/.ssh/kw26_pool'])
pathlib.Path('/content/tunnel.sh').write_text(f'''while true; do
  ssh -N -i /root/.ssh/kw26_pool -p 10022 -o StrictHostKeyChecking=yes -o IdentitiesOnly=yes \\
      -o ExitOnForwardFailure=yes -o ServerAliveInterval=15 -o ServerAliveCountMax=3 {forwards} outofmemory@123.37.28.167
  echo "$(date -u +%FT%TZ) tunnel exited $?"; sleep 5
done
''')
open('/content/tunnel.log', 'w').close()
subprocess.Popen(['bash', '/content/tunnel.sh'], stdout=open('/content/tunnel.log', 'ab'),
                 stderr=subprocess.STDOUT, start_new_session=True)
print('account', ACCOUNT, '| tunnel started:', forwards, flush=True)

def last_rate(port):
    lines = [l for l in open(f'/content/sglang-{port}.log', errors='replace').read()[-8000:].splitlines() if 'gen throughput' in l]
    if not lines:
        return '-'
    return (lines[-1].split('#running-req:')[1].split(',')[0].strip() + 'req/'
            + lines[-1].split('gen throughput (token/s):')[1].split(',')[0].strip() + 'tok/s')

while True:
    time.sleep(60)
    for port in PORTS:                       # 서버가 죽었으면 다시 올린다
        if not healthy(port):
            print(time.strftime('%H:%M:%S'), port, 'DOWN → restart', flush=True)
            try:
                start_server(port)
            except Exception as exc:
                print('restart failed:', str(exc)[:200], flush=True)
    ssh_up = bool(subprocess.run(['pgrep', '-f', 'ssh -N -i /root/.ssh/kw26_pool'], capture_output=True).stdout)
    tail = open('/content/tunnel.log', errors='replace').read()[-400:]
    why = '키 거부됨 — 서버 등록 대기(KW26_PUBKEY)' if 'Permission denied' in tail else ('포트 거부 — 계정 번호 확인' if 'forwarding failed' in tail else 'DOWN(재시도 중)')
    print(time.strftime('%H:%M:%S'), 'health', {p: healthy(p) for p in PORTS},
          '| tunnel', 'UP' if ssh_up else why, '|', [last_rate(p) for p in PORTS],
          '| gpu', sh(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used', '--format=csv,noheader'], check=False).strip(), flush=True)